<a href="https://colab.research.google.com/github/KaanCesur354/CENG467_TakeHome/blob/main/Ceng467_TakeHome.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Gerekli kütüphaneleri kur
!pip install datasets transformers torch scikit-learn matplotlib seaborn -q

import random
import numpy as np
import torch

# Tekrarlanabilirlik için seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Kurulum tamam!")
print("GPU var mı:", torch.cuda.is_available())
print("Seed:", SEED)

Kurulum tamam!
GPU var mı: True
Seed: 42


In [3]:
# Load IMDb dataset
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)
print("\n--- Sample training record ---")
print("Text:", dataset["train"][0]["text"][:300], "...")
print("Label:", dataset["train"][0]["label"], "(0=Negative, 1=Positive)")
print("\nTotal training examples:", len(dataset["train"]))
print("Total test examples:", len(dataset["test"]))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

--- Sample training record ---
Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...
Label: 0 (0=Negative, 1=Positive)

Total training examples: 25000
Total test examples: 25000


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Convert to pandas dataframe
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

# Carve out a validation set from training data (80/20 split)
train_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df["label"]
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

# Check label distribution
print("\nTrain label distribution:")
print(train_df["label"].value_counts())

Train size: 20000
Validation size: 5000
Test size: 25000

Train label distribution:
label
1    10000
0    10000
Name: count, dtype: int64


In [5]:
import re
import string

def preprocess_text(text, remove_stopwords=False):
    # Lowercase
    text = text.lower()
    # Remove HTML tags (IMDb reviews often contain <br /> tags)
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    if remove_stopwords:
        from nltk.corpus import stopwords
        stop_words = set(stopwords.words('english'))
        text = ' '.join([w for w in text.split() if w not in stop_words])

    return text

# Apply preprocessing — two versions for comparison
# Version 1: with stopwords kept
train_df["text_clean"] = train_df["text"].apply(lambda x: preprocess_text(x, remove_stopwords=False))
val_df["text_clean"] = val_df["text"].apply(lambda x: preprocess_text(x, remove_stopwords=False))
test_df["text_clean"] = test_df["text"].apply(lambda x: preprocess_text(x, remove_stopwords=False))

print("Preprocessing done!")
print("\nOriginal text sample:")
print(train_df["text"].iloc[0][:200])
print("\nCleaned text sample:")
print(train_df["text_clean"].iloc[0][:200])

Preprocessing done!

Original text sample:
I have always been a huge James Bond fanatic! I have seen almost all of the films except for Die Another Day, and The World Is Not Enough. The graphic's for Everything Or Nothing are breathtaking! The

Cleaned text sample:
i have always been a huge james bond fanatic i have seen almost all of the films except for die another day and the world is not enough the graphics for everything or nothing are breathtaking the voic


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# TF-IDF Vectorization
print("Fitting TF-IDF vectorizer...")
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),  # unigrams + bigrams
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(train_df["text_clean"])
X_val_tfidf = tfidf.transform(val_df["text_clean"])
X_test_tfidf = tfidf.transform(test_df["text_clean"])

print("TF-IDF shape:", X_train_tfidf.shape)

# Train Logistic Regression
print("\nTraining Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
lr_model.fit(X_train_tfidf, train_df["label"])

# Evaluate on validation set
val_preds_lr = lr_model.predict(X_val_tfidf)
val_acc_lr = accuracy_score(val_df["label"], val_preds_lr)
val_f1_lr = f1_score(val_df["label"], val_preds_lr, average="macro")

print("\n--- Logistic Regression (TF-IDF) Results ---")
print(f"Validation Accuracy: {val_acc_lr:.4f}")
print(f"Validation Macro-F1: {val_f1_lr:.4f}")
print("\nClassification Report:")
print(classification_report(val_df["label"], val_preds_lr, target_names=["Negative", "Positive"]))

Fitting TF-IDF vectorizer...
TF-IDF shape: (20000, 50000)

Training Logistic Regression...

--- Logistic Regression (TF-IDF) Results ---
Validation Accuracy: 0.8964
Validation Macro-F1: 0.8964

Classification Report:
              precision    recall  f1-score   support

    Negative       0.90      0.89      0.90      2500
    Positive       0.89      0.90      0.90      2500

    accuracy                           0.90      5000
   macro avg       0.90      0.90      0.90      5000
weighted avg       0.90      0.90      0.90      5000



In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# ---- Vocabulary Builder ----
def build_vocab(texts, max_vocab=30000):
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_df["text_clean"])
print(f"Vocabulary size: {len(vocab)}")

# ---- Dataset ----
class IMDbDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=256):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def encode(self, text):
        tokens = text.split()[:self.max_len]
        ids = [self.vocab.get(t, 1) for t in tokens]
        # Pad or truncate
        ids += [0] * (self.max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(self.encode(self.texts.iloc[idx]), dtype=torch.long)
        y = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return x, y

train_dataset = IMDbDataset(train_df["text_clean"], train_df["label"], vocab)
val_dataset   = IMDbDataset(val_df["text_clean"],   val_df["label"],   vocab)
test_dataset  = IMDbDataset(test_df["text_clean"],  test_df["label"],  vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64)
test_loader  = DataLoader(test_dataset,  batch_size=64)

print("Datasets ready!")

Vocabulary size: 30000
Datasets ready!


In [8]:
# ---- BiLSTM Model ----
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super(BiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 2)  # *2 because bidirectional

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, (hidden, _) = self.lstm(embedded)
        # Concatenate last forward and backward hidden states
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(hidden))

# ---- Training Function ----
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (preds.argmax(1) == y).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x)
            loss = criterion(preds, y)
            total_loss += loss.item()
            correct += (preds.argmax(1) == y).sum().item()
            all_preds.extend(preds.argmax(1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return total_loss / len(loader), correct / len(loader.dataset), all_preds, all_labels

# ---- Train BiLSTM ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

bilstm_model = BiLSTM(vocab_size=len(vocab)).to(device)
optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
print("\nTraining BiLSTM...")
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(bilstm_model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = evaluate(bilstm_model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Using device: cuda

Training BiLSTM...
Epoch 1/5 | Train Loss: 0.6917 | Train Acc: 0.5274 | Val Loss: 0.6917 | Val Acc: 0.5006
Epoch 2/5 | Train Loss: 0.6702 | Train Acc: 0.5818 | Val Loss: 0.6335 | Val Acc: 0.6222
Epoch 3/5 | Train Loss: 0.5435 | Train Acc: 0.7295 | Val Loss: 0.4017 | Val Acc: 0.8200
Epoch 4/5 | Train Loss: 0.3774 | Train Acc: 0.8350 | Val Loss: 0.3161 | Val Acc: 0.8656
Epoch 5/5 | Train Loss: 0.2858 | Train Acc: 0.8825 | Val Loss: 0.2951 | Val Acc: 0.8722


In [9]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Evaluate BiLSTM on validation set
_, val_acc_bilstm, val_preds_bilstm, val_labels_bilstm = evaluate(
    bilstm_model, val_loader, criterion, device
)
val_f1_bilstm = f1_score(val_labels_bilstm, val_preds_bilstm, average="macro")

print("--- BiLSTM Results ---")
print(f"Validation Accuracy: {val_acc_bilstm:.4f}")
print(f"Validation Macro-F1: {val_f1_bilstm:.4f}")
print("\nClassification Report:")
print(classification_report(val_labels_bilstm, val_preds_bilstm, target_names=["Negative", "Positive"]))

--- BiLSTM Results ---
Validation Accuracy: 0.8722
Validation Macro-F1: 0.8722

Classification Report:
              precision    recall  f1-score   support

    Negative       0.88      0.86      0.87      2500
    Positive       0.86      0.88      0.87      2500

    accuracy                           0.87      5000
   macro avg       0.87      0.87      0.87      5000
weighted avg       0.87      0.87      0.87      5000



In [10]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader

# ---- Tokenizer ----
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# ---- Dataset ----
class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

print("Tokenizing datasets...")
bert_train = BertDataset(train_df["text_clean"], train_df["label"], tokenizer)
bert_val   = BertDataset(val_df["text_clean"],   val_df["label"],   tokenizer)
bert_test  = BertDataset(test_df["text_clean"],  test_df["label"],  tokenizer)

bert_train_loader = DataLoader(bert_train, batch_size=32, shuffle=True)
bert_val_loader   = DataLoader(bert_val,   batch_size=32)
bert_test_loader  = DataLoader(bert_test,  batch_size=32)

print("Tokenization done!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing datasets...
Tokenization done!


In [11]:
from transformers import DistilBertForSequenceClassification
from torch.optim import AdamW

# Load pretrained DistilBERT
print("Loading DistilBERT model...")
bert_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
).to(device)

optimizer_bert = AdamW(bert_model.parameters(), lr=2e-5)

# ---- Train & Evaluate Functions for BERT ----
def train_bert_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct = 0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate_bert(model, loader, device):
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / len(loader.dataset), all_preds, all_labels

# ---- Training ----
EPOCHS_BERT = 3
print("\nTraining DistilBERT...")
for epoch in range(EPOCHS_BERT):
    train_loss, train_acc = train_bert_epoch(bert_model, bert_train_loader, optimizer_bert, device)
    val_loss, val_acc, _, _ = evaluate_bert(bert_model, bert_val_loader, device)
    print(f"Epoch {epoch+1}/{EPOCHS_BERT} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Loading DistilBERT model...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training DistilBERT...
Epoch 1/3 | Train Loss: 0.3138 | Train Acc: 0.8637 | Val Loss: 0.2359 | Val Acc: 0.9008
Epoch 2/3 | Train Loss: 0.1793 | Train Acc: 0.9351 | Val Loss: 0.2229 | Val Acc: 0.9128
Epoch 3/3 | Train Loss: 0.1027 | Train Acc: 0.9652 | Val Loss: 0.2654 | Val Acc: 0.9102


In [12]:
# Final evaluation of DistilBERT on validation set
_, val_acc_bert, val_preds_bert, val_labels_bert = evaluate_bert(
    bert_model, bert_val_loader, device
)
val_f1_bert = f1_score(val_labels_bert, val_preds_bert, average="macro")

print("--- DistilBERT Results ---")
print(f"Validation Accuracy: {val_acc_bert:.4f}")
print(f"Validation Macro-F1: {val_f1_bert:.4f}")
print("\nClassification Report:")
print(classification_report(val_labels_bert, val_preds_bert, target_names=["Negative", "Positive"]))

# ---- Summary Table ----
print("\n" + "="*55)
print(f"{'Model':<25} {'Accuracy':>10} {'Macro-F1':>10}")
print("="*55)
print(f"{'TF-IDF + LogReg':<25} {val_acc_lr:>10.4f} {val_f1_lr:>10.4f}")
print(f"{'BiLSTM':<25} {val_acc_bilstm:>10.4f} {val_f1_bilstm:>10.4f}")
print(f"{'DistilBERT':<25} {val_acc_bert:>10.4f} {val_f1_bert:>10.4f}")
print("="*55)


--- DistilBERT Results ---
Validation Accuracy: 0.9102
Validation Macro-F1: 0.9102

Classification Report:
              precision    recall  f1-score   support

    Negative       0.91      0.91      0.91      2500
    Positive       0.91      0.91      0.91      2500

    accuracy                           0.91      5000
   macro avg       0.91      0.91      0.91      5000
weighted avg       0.91      0.91      0.91      5000


Model                       Accuracy   Macro-F1
TF-IDF + LogReg               0.8964     0.8964
BiLSTM                        0.8722     0.8722
DistilBERT                    0.9102     0.9102


In [13]:
# Misclassification analysis for DistilBERT
import pandas as pd

val_df_reset = val_df.reset_index(drop=True)

results_df = pd.DataFrame({
    "text": val_df_reset["text"],
    "true_label": val_labels_bert,
    "pred_label": val_preds_bert
})

# Filter misclassified examples
misclassified = results_df[results_df["true_label"] != results_df["pred_label"]].reset_index(drop=True)

print(f"Total misclassified: {len(misclassified)} / {len(val_df)} ({len(misclassified)/len(val_df)*100:.1f}%)")
print("\n" + "="*80)

label_map = {0: "Negative", 1: "Positive"}

for i in range(5):
    row = misclassified.iloc[i]
    print(f"\n--- Example {i+1} ---")
    print(f"True Label : {label_map[row['true_label']]}")
    print(f"Predicted  : {label_map[row['pred_label']]}")
    print(f"Text       : {row['text'][:400]}...")
    print("="*80)

Total misclassified: 449 / 5000 (9.0%)


--- Example 1 ---
True Label : Positive
Predicted  : Negative
Text       : Yowsa! If you REALLY want some ACTION, check out the babes and bombs on this non-stop thriller! Veteran star MARTIN SHEEN leads a trio of supermodels on a mission to stop nuclear terrorism... but director Dean Hamilton doesn't let this heavy plotline get in the way of massive doses of TEENSY-SWIMSUIT scenes, jiggly beach jogs, hubba-hubba hot tubs and the like! Want action? You'll get more of it h...

--- Example 2 ---
True Label : Positive
Predicted  : Negative
Text       : Any movie that shows federal PIGs (Persons In Government) to be the power-mad threats they are in real life has a lot to recommend it to me.<br /><br />Alas, the script supervision and editing and even, at times, the directing are flawed so there will be people who will disparage the whole movie and ignore the good moments.<br /><br />I saw the original way back when it was new and hated it, despi...


In [14]:
# Final evaluation on TEST SET (used only once!)
print("="*55)
print("FINAL TEST SET EVALUATION")
print("="*55)

# TF-IDF + LogReg
test_preds_lr = lr_model.predict(X_test_tfidf)
test_acc_lr = accuracy_score(test_df["label"], test_preds_lr)
test_f1_lr = f1_score(test_df["label"], test_preds_lr, average="macro")

# BiLSTM
_, test_acc_bilstm, test_preds_bilstm, test_labels = evaluate(
    bilstm_model, test_loader, criterion, device
)
test_f1_bilstm = f1_score(test_labels, test_preds_bilstm, average="macro")

# DistilBERT
_, test_acc_bert, test_preds_bert, test_labels_bert = evaluate_bert(
    bert_model, bert_test_loader, device
)
test_f1_bert = f1_score(test_labels_bert, test_preds_bert, average="macro")

print(f"\n{'Model':<25} {'Accuracy':>10} {'Macro-F1':>10}")
print("="*55)
print(f"{'TF-IDF + LogReg':<25} {test_acc_lr:>10.4f} {test_f1_lr:>10.4f}")
print(f"{'BiLSTM':<25} {test_acc_bilstm:>10.4f} {test_f1_bilstm:>10.4f}")
print(f"{'DistilBERT':<25} {test_acc_bert:>10.4f} {test_f1_bert:>10.4f}")
print("="*55)


FINAL TEST SET EVALUATION

Model                       Accuracy   Macro-F1
TF-IDF + LogReg               0.8913     0.8913
BiLSTM                        0.8622     0.8622
DistilBERT                    0.9094     0.9094
